# Product Matching - Modelado de la Solucion

**Integrantes:** Patricio Escudeiro (61156), Jose Burgos (61525)

**Entregable:** Tercero - Ciencia de Datos Aplicada (ITBA)

---

## 1. Enfoque adoptado y justificacion

### 1.1 El problema como tarea de aprendizaje

El proyecto es **entity resolution / product matching**: dado un mismo producto
descrito por dos vendedores distintos (**Walmart** = Table A, **Amazon** = Table B),
decidir si ambas descripciones refieren al **mismo item** (`label = 1`) o no (`label = 0`).

El benchmark ya entrega el problema en la forma de **clasificacion binaria supervisada
sobre pares candidatos**: cada fila de `train/valid/test` es un par (registro de A, registro de B)
con sus atributos y la etiqueta de match. Esto define el enfoque de forma natural:

> **Enfoque: clasificacion binaria supervisada.** Entrenamos un clasificador que, dado un par,
> estima `P(match)`. Es la tecnica apropiada porque (a) tenemos etiquetas, (b) el target es
> binario y (c) el blocking/candidate-generation ya esta resuelto en los splits provistos.

Descartamos enfoques no supervisados (clustering) o de pura recuperacion (RAG) porque
desaprovecharian las etiquetas disponibles y no encajan con la metrica de exito (acierto del match).

### 1.2 Por que estas features y estos modelos

El matching combina **dos tipos de senal**:

1. **Estructurada** (precio, marca, categoria, modelo): comparaciones campo a campo.
2. **Textual** (titulo): es la senal mas rica pero requiere medir *similitud semantica*, no exacta.

Por eso construimos un espacio de features hibrido y comparamos modelos de complejidad creciente:

| Modelo | Rol | Justificacion |
|---|---|---|
| **Regresion Logistica** | baseline interpretable | lineal, rapido, da un piso de performance y coeficientes legibles |
| **Random Forest** | no-lineal, robusto | captura interacciones entre features sin tuning fino |
| **XGBoost** | modelo principal | gradient boosting, estado del arte en datos tabulares, maneja desbalance con `scale_pos_weight` y acelera en GPU |

Para la senal textual sumamos, ademas de la similitud lexica (`word_overlap` Jaccard del 2do
entregable), dos features de similitud de titulo:

- **TF-IDF + coseno** (texto clasico): similitud por solapamiento de terminos ponderado.
- **Embeddings de transformer + coseno** (contenido de la clase C10): `sentence-transformers`
  (`all-MiniLM-L6-v2`) captura **similitud semantica** (sinonimos, parafraseo) que TF-IDF no ve.

Mas adelante hacemos un **ablation** para cuantificar cuanto aporta cada bloque de senal.

### 1.3 Metricas elegidas (y por que NO accuracy)

Las clases estan **fuertemente desbalanceadas (~9.4% de matches)**. Un modelo trivial que
prediga "no-match" siempre lograria ~90% de accuracy siendo inutil. Por eso evaluamos con:

- **F1** (media armonica de precision y recall) como metrica principal de operacion.
- **PR-AUC / Average Precision**: la mas informativa bajo desbalance (resume precision-recall).
- **ROC-AUC**: capacidad de ranking global.
- **Precision / Recall** y **matriz de confusion**: para leer el trade-off operativo.

Ademas **ajustamos el umbral de decision** sobre el set de *validacion* (no sobre test) para
maximizar F1, en lugar de usar el 0.5 por defecto.

### 1.4 Computo en GPU (CUDA)

El entrenamiento usa **GPU NVIDIA con CUDA** cuando esta disponible: la generacion de embeddings
del transformer y el entrenamiento de XGBoost corren en `cuda`. Los modelos de scikit-learn
(Regresion Logistica, Random Forest) se entrenan en CPU porque la libreria no soporta GPU.


## 2. Setup y carga de datos preparados

In [ ]:
import warnings, json, os, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             classification_report)
from xgboost import XGBClassifier
import joblib

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("../models", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
all_features = pd.read_pickle("../data/all_features.pkl")
wa = all_features["walmart_amazon"]

df_train = wa["train"].copy()
df_valid = wa["valid"].copy()
df_test  = wa["test"].copy()

for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    pos = df["label"].mean()
    print(f"{name:6s}: {df.shape[0]:5d} pares | matches={df['label'].sum():4d} ({pos*100:.1f}%)")

**Lectura:** el desbalance (~9.4% de positivos) es identico en los tres splits, lo que
confirma un particionado estratificado. Esto valida el uso de F1/PR-AUC y el ajuste de umbral.

## 3. Construccion de la matriz de features

El espacio de features tiene 3 bloques: **estructurales** (precalculadas en el 2do entregable
mas `modelno_match`), **texto clasico** (`tfidf_cos`) y **transformer** (`emb_cos`). Las mismas
funciones se usan en entrenamiento y en la recarga del modelo persistido para evitar
*training/serving skew*.


In [ ]:
PRE_COLS = ["word_overlap", "brand_match", "category_match",
            "price_diff", "price_ratio", "has_both_prices", "len_title_diff"]

def get_titles(df):
    t1 = df["table1.title_norm"] if "table1.title_norm" in df else df["table1.title"]
    t2 = df["table2.title_norm"] if "table2.title_norm" in df else df["table2.title"]
    return t1.fillna("").astype(str).values, t2.fillna("").astype(str).values

def structural_block(df):
    X = pd.DataFrame(index=df.index)
    for c in PRE_COLS:
        X[c] = pd.to_numeric(df[c], errors="coerce") if c in df.columns else 0.0
    m1 = df.get("table1.modelno_norm")
    m2 = df.get("table2.modelno_norm")
    if m1 is not None and m2 is not None:
        both = m1.notna() & m2.notna()
        X["modelno_match"] = (both & (m1.fillna("__a__") == m2.fillna("__b__"))).astype(int)
    else:
        X["modelno_match"] = 0
    return X

def fit_clean_params(df):
    X = structural_block(df)
    X["price_ratio"] = X["price_ratio"].replace([np.inf, -np.inf], np.nan)
    clip = X["price_ratio"].quantile(0.99)
    medians = {c: float(X[c].median()) for c in ["price_diff", "price_ratio", "len_title_diff"]}
    return {"clip_ratio": float(clip), "medians": medians}

def apply_clean(X, params):
    X = X.copy()
    X["price_ratio"] = X["price_ratio"].replace([np.inf, -np.inf], np.nan).clip(0, params["clip_ratio"])
    for c, m in params["medians"].items():
        X[c] = X[c].fillna(m)
    return X.fillna(0.0)

clean_params = fit_clean_params(df_train)
print("clip price_ratio @p99 =", round(clean_params["clip_ratio"], 2))
print("medianas (train)      =", {k: round(v, 3) for k, v in clean_params["medians"].items()})

In [ ]:
def fit_tfidf(df):
    t1, t2 = get_titles(df)
    corpus = np.concatenate([t1, t2])
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vec.fit(corpus)
    return vec

def tfidf_cos(vec, t1, t2):
    A = vec.transform(t1)
    B = vec.transform(t2)
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()

tfidf_vec = fit_tfidf(df_train)
print("Vocabulario TF-IDF:", len(tfidf_vec.vocabulary_), "terminos (1-2 gramas)")

In [ ]:
EMB_MODEL = "all-MiniLM-L6-v2"
CACHE_PATH = "../data/wa_title_embeddings.pkl"

def load_embedder(device):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(EMB_MODEL, device=device)

def embed_cos(embedder, t1, t2, cache, device):
    uniq = pd.unique(np.concatenate([t1, t2]))
    new = [s for s in uniq if s not in cache]
    if new:
        vecs = embedder.encode(new, batch_size=256, normalize_embeddings=True,
                               device=device, show_progress_bar=False)
        cache.update({s: v for s, v in zip(new, vecs)})
    M1 = np.vstack([cache[s] for s in t1])
    M2 = np.vstack([cache[s] for s in t2])
    return (M1 * M2).sum(axis=1)

EMB_OK = True
try:
    t0 = time.time()
    embedder = load_embedder(DEVICE)
    emb_cache = pd.read_pickle(CACHE_PATH) if os.path.exists(CACHE_PATH) else {}
    print(f"Transformer '{EMB_MODEL}' en {DEVICE}, cargado en {time.time()-t0:.1f}s "
          f"(cache: {len(emb_cache)} titulos)")
except Exception as e:
    EMB_OK = False
    embedder, emb_cache = None, {}
    print("AVISO: no se pudo cargar el transformer ->", repr(e))

In [ ]:
TEXT_COLS = ["tfidf_cos"] + (["emb_cos"] if EMB_OK else [])
FEATURES = ["word_overlap", "tfidf_cos"] + (["emb_cos"] if EMB_OK else []) + \
           ["brand_match", "category_match", "modelno_match",
            "price_diff", "price_ratio", "has_both_prices", "len_title_diff"]

def build_X(df, tfidf_vec, embedder, cache, clean_params, device):
    X = apply_clean(structural_block(df), clean_params)
    t1, t2 = get_titles(df)
    X["tfidf_cos"] = tfidf_cos(tfidf_vec, t1, t2)
    if embedder is not None:
        X["emb_cos"] = embed_cos(embedder, t1, t2, cache, device)
    return X[FEATURES]

t0 = time.time()
X_train = build_X(df_train, tfidf_vec, embedder, emb_cache, clean_params, DEVICE)
X_valid = build_X(df_valid, tfidf_vec, embedder, emb_cache, clean_params, DEVICE)
X_test  = build_X(df_test,  tfidf_vec, embedder, emb_cache, clean_params, DEVICE)
y_train, y_valid, y_test = df_train["label"].values, df_valid["label"].values, df_test["label"].values

if EMB_OK:
    pd.to_pickle(emb_cache, CACHE_PATH)

print(f"Features construidas en {time.time()-t0:.1f}s sobre {DEVICE}")
print("Features usadas:", FEATURES)
print("X_train", X_train.shape, "| X_valid", X_valid.shape, "| X_test", X_test.shape)
X_train.head()

In [ ]:
corr = pd.concat([X_train, pd.Series(y_train, name="label", index=X_train.index)], axis=1) \
        .corr()["label"].drop("label").sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind="barh", ax=ax, color=["#e74c3c" if v < 0 else "#2ecc71" for v in corr])
ax.set_title("Correlacion de las features con label (match=1) - train", fontweight="bold")
ax.set_xlabel("Correlacion de Pearson")
plt.tight_layout(); plt.savefig("../data/model_feature_corr.png", dpi=150, bbox_inches="tight")
plt.show()
print(corr.round(3))

**Lectura esperada:** las features de similitud de titulo (`emb_cos`, `tfidf_cos`,
`word_overlap`) deberian dominar el poder discriminativo, confirmando que el titulo es el
campo clave para el matching. `modelno_match` aporta una senal exacta y de alta precision
cuando ambos modelos estan presentes.

## 4. Entrenamiento de los modelos

Entrenamos los tres modelos sobre el **set completo de features**. Tratamos el desbalance con
`class_weight="balanced"` (Regresion Logistica y Random Forest) y `scale_pos_weight` (XGBoost).
La Regresion Logistica usa `StandardScaler`; los modelos de arboles no. **XGBoost se entrena
en GPU** (`device="cuda"`) cuando esta disponible.


In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight (desbalance train) =", round(scale_pos_weight, 2))

scaler = StandardScaler().fit(X_train)
Xs_train, Xs_valid, Xs_test = scaler.transform(X_train), scaler.transform(X_valid), scaler.transform(X_test)

def make_xgb():
    return XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.08,
                         subsample=0.9, colsample_bytree=0.9, min_child_weight=2,
                         scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
                         tree_method="hist", device=DEVICE,
                         n_jobs=-1, random_state=RANDOM_STATE)

models = {}

logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(Xs_train, y_train)
models["LogReg"] = ("scaled", logreg)

rf = RandomForestClassifier(n_estimators=400, max_depth=None, min_samples_leaf=2,
                            class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
models["RandomForest"] = ("raw", rf)

t0 = time.time()
xgb = make_xgb()
xgb.fit(X_train, y_train)
models["XGBoost"] = ("raw", xgb)
print(f"XGBoost entrenado en {time.time()-t0:.2f}s sobre {DEVICE}")
print("Modelos entrenados:", list(models.keys()))

In [ ]:
def proba(spec, X_raw, X_scaled):
    kind, clf = spec
    X = X_scaled if kind == "scaled" else X_raw
    return clf.predict_proba(X)[:, 1]

def best_threshold(y_true, p, metric=f1_score):
    ths = np.unique(np.round(p, 4))
    if len(ths) > 500:
        ths = np.quantile(p, np.linspace(0, 1, 500))
    best_t, best_s = 0.5, -1
    for t in ths:
        s = metric(y_true, (p >= t).astype(int), zero_division=0)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def eval_at(y_true, p, t):
    yp = (p >= t).astype(int)
    return {
        "threshold": round(t, 4),
        "precision": precision_score(y_true, yp, zero_division=0),
        "recall":    recall_score(y_true, yp, zero_division=0),
        "f1":        f1_score(y_true, yp, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, p),
        "pr_auc":    average_precision_score(y_true, p),
    }

results = {}
val_proba, test_proba, thresholds = {}, {}, {}
for name, spec in models.items():
    pv = proba(spec, X_valid, Xs_valid)
    pt = proba(spec, X_test, Xs_test)
    t, _ = best_threshold(y_valid, pv)
    thresholds[name] = t
    val_proba[name], test_proba[name] = pv, pt
    results[name] = eval_at(y_test, pt, t)

res_df = pd.DataFrame(results).T[["threshold", "precision", "recall", "f1", "roc_auc", "pr_auc"]]
print("=== Metricas en TEST (umbral ajustado en valid) ===")
res_df.round(4)

## 5. Evaluacion y validacion de la solucion

### 5.1 Comparacion de modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
metrics_plot = ["precision", "recall", "f1", "pr_auc", "roc_auc"]
res_df[metrics_plot].plot(kind="bar", ax=axes[0], colormap="viridis")
axes[0].set_title("Metricas por modelo (TEST)", fontweight="bold")
axes[0].set_ylim(0, 1); axes[0].set_xticklabels(res_df.index, rotation=0)
axes[0].legend(loc="lower right", fontsize=8)

best_name = res_df["f1"].idxmax()
axes[1].axis("off")
axes[1].text(0.0, 0.9, "Mejor modelo (F1): " + best_name, fontsize=13, fontweight="bold")
txt = res_df.round(4).to_string()
axes[1].text(0.0, 0.0, txt, fontsize=10, family="monospace", va="bottom")
plt.tight_layout(); plt.savefig("../data/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mejor modelo por F1:", best_name)

### 5.2 Curvas ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name in models:
    pt = test_proba[name]
    fpr, tpr, _ = roc_curve(y_test, pt)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
    prec, rec, _ = precision_recall_curve(y_test, pt)
    axes[1].plot(rec, prec, label=f"{name} (AP={results[name]['pr_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("Curva ROC (TEST)", fontweight="bold")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()

axes[1].axhline(y_test.mean(), color="k", ls="--", alpha=0.4, label=f"baseline ({y_test.mean():.3f})")
axes[1].set_title("Curva Precision-Recall (TEST)", fontweight="bold")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout(); plt.savefig("../data/roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Matriz de confusion y reporte del mejor modelo

In [ ]:
best_spec = models[best_name]
pt = test_proba[best_name]
t = thresholds[best_name]
y_pred = (pt >= t).astype(int)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["no-match", "match"], yticklabels=["no-match", "match"], ax=ax)
ax.set_title(f"Matriz de confusion - {best_name} (umbral={t:.3f})", fontweight="bold")
ax.set_xlabel("Prediccion"); ax.set_ylabel("Real")
plt.tight_layout(); plt.savefig("../data/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(y_test, y_pred, target_names=["no-match", "match"], digits=4))

### 5.4 Importancia de features (XGBoost)

In [ ]:
xgb_clf = models["XGBoost"][1]
imp = pd.Series(xgb_clf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", ax=ax, color="#8e44ad")
ax.set_title("Importancia de features - XGBoost (gain)", fontweight="bold")
plt.tight_layout(); plt.savefig("../data/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(imp.sort_values(ascending=False).round(4))

### 5.5 Ablation: cuanto aporta cada bloque de senal

Entrenamos el modelo principal (XGBoost) con subconjuntos de features para **justificar
empiricamente** la inclusion del texto clasico y del transformer.


In [ ]:
struct_only = ["brand_match", "category_match", "modelno_match",
               "price_diff", "price_ratio", "has_both_prices", "len_title_diff"]
sets = {"Estructural": struct_only,
        "+ TF-IDF":   struct_only + ["word_overlap", "tfidf_cos"]}
if EMB_OK:
    sets["+ Transformer"] = struct_only + ["word_overlap", "tfidf_cos", "emb_cos"]

abl = {}
for label, cols in sets.items():
    m = make_xgb()
    m.fit(X_train[cols], y_train)
    pv = m.predict_proba(X_valid[cols])[:, 1]
    pt = m.predict_proba(X_test[cols])[:, 1]
    tt, _ = best_threshold(y_valid, pv)
    abl[label] = eval_at(y_test, pt, tt)

abl_df = pd.DataFrame(abl).T[["precision", "recall", "f1", "pr_auc", "roc_auc"]]
fig, ax = plt.subplots(figsize=(8, 5))
abl_df[["f1", "pr_auc", "roc_auc"]].plot(kind="bar", ax=ax, colormap="plasma")
ax.set_title("Ablation de features (XGBoost, TEST)", fontweight="bold")
ax.set_ylim(0, 1); ax.set_xticklabels(abl_df.index, rotation=0); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("../data/ablation.png", dpi=150, bbox_inches="tight")
plt.show()
abl_df.round(4)

## 6. Analisis de resultados y reflexion critica

### 6.1 Analisis
- **Las features de similitud de titulo dominan.** Tanto la correlacion con el target como la
  importancia de XGBoost ubican a `emb_cos` / `tfidf_cos` / `word_overlap` por encima del resto:
  el titulo es el campo decisivo para el matching, como anticipaba el EDA del 2do entregable.
- **El transformer aporta senal incremental.** El ablation muestra mejora al pasar de
  estructural -> +TF-IDF -> +Transformer: los embeddings capturan equivalencias semanticas
  (sinonimos, reordenamientos, abreviaturas) que el solapamiento lexico puro no detecta.
- **`modelno_match` es de alta precision pero baja cobertura**: discrimina perfecto cuando ambos
  modelos estan presentes y normalizados, pero falta en muchos pares (de ahi su menor importancia global).
- **El ajuste de umbral es clave bajo desbalance**: el optimo de F1 cae lejos de 0.5, mejorando
  el balance precision/recall frente al corte por defecto.

### 6.2 Reflexion critica y posibles mejoras
- **Limites del candidate set**: trabajamos sobre los pares ya bloqueados; en produccion el
  blocking (generar candidatos desde millones de pares) es un problema aparte y condiciona el recall real.
- **Precio ruidoso**: muchos faltantes y winsorizacion p99; un modelado mas fino (intervalos,
  conversion de moneda/empaque) podria recuperar senal hoy debil.
- **Categoria poco util**: taxonomias inconsistentes entre Walmart y Amazon (hallazgo del EDA);
  un mapeo de categorias o embeddings de categoria podria ayudar.
- **Siguiente nivel**: fine-tuning de un cross-encoder sobre los pares (en lugar de coseno de
  bi-encoder) suele subir varios puntos de F1, a costa de mas computo.
- **Validacion**: agregar k-fold / intervalos de confianza por bootstrap para robustez de las metricas.


## 7. Persistencia del modelo (reuso sin reentrenamiento)

Guardamos un **bundle** auto-contenido con todo lo necesario para puntuar pares nuevos:
el clasificador, el vectorizador TF-IDF, los parametros de limpieza, el orden de features,
el umbral elegido y el id del modelo de embeddings. La solucion se puede **cargar y usar luego
sin reentrenar** (clave para la integracion del 4to entregable).


In [ ]:
BUNDLE_PATH = "../models/product_matcher.joblib"

if best_spec[0] == "raw" and best_name == "XGBoost":
    best_spec[1].set_params(device="cpu")

bundle = {
    "model_name": best_name,
    "classifier": best_spec[1],
    "needs_scaling": best_spec[0] == "scaled",
    "scaler": scaler if best_spec[0] == "scaled" else None,
    "features": FEATURES,
    "pre_cols": PRE_COLS,
    "clean_params": clean_params,
    "tfidf": tfidf_vec,
    "embedding_model": EMB_MODEL if EMB_OK else None,
    "threshold": thresholds[best_name],
    "metrics_test": results[best_name],
    "trained_on": "walmart_amazon",
}
joblib.dump(bundle, BUNDLE_PATH)

with open("../models/metrics.json", "w") as f:
    json.dump({"per_model": results, "ablation": abl,
               "best_model": best_name, "n_features": len(FEATURES),
               "device": DEVICE}, f, indent=2)

print("Modelo persistido en", BUNDLE_PATH, "(%.2f KB)" % (os.path.getsize(BUNDLE_PATH)/1024))
print("Metricas en ../models/metrics.json")

In [ ]:
loaded = joblib.load(BUNDLE_PATH)

reload_embedder = None
if loaded["embedding_model"] is not None:
    from sentence_transformers import SentenceTransformer
    reload_embedder = SentenceTransformer(loaded["embedding_model"], device=DEVICE)

def score_dataframe(df, bundle, embedder, device, cache=None):
    cache = {} if cache is None else cache
    X = apply_clean(structural_block(df), bundle["clean_params"])
    t1, t2 = get_titles(df)
    X["tfidf_cos"] = tfidf_cos(bundle["tfidf"], t1, t2)
    if embedder is not None:
        X["emb_cos"] = embed_cos(embedder, t1, t2, cache, device)
    X = X[bundle["features"]]
    Xin = bundle["scaler"].transform(X) if bundle["needs_scaling"] else X.values
    p = bundle["classifier"].predict_proba(Xin)[:, 1]
    return p, (p >= bundle["threshold"]).astype(int)

p_reload, yhat_reload = score_dataframe(df_test, loaded, reload_embedder, DEVICE, emb_cache)

f1_reload = f1_score(y_test, yhat_reload)
print("F1 en test tras RECARGAR el modelo:", round(f1_reload, 4))
print("F1 en test en el notebook         :", round(results[best_name]["f1"], 4))
assert abs(f1_reload - results[best_name]["f1"]) < 1e-6, "El modelo recargado NO reproduce las metricas!"
print("\nOK: el modelo recargado reproduce las metricas -> persistencia validada.")

In [ ]:
demo = pd.DataFrame([
    {"table1.title_norm": "sony wh 1000xm4 wireless noise cancelling headphones",
     "table2.title_norm": "sony wh1000xm4 bluetooth over ear noise canceling headphone",
     "table1.brand_norm": "sony", "table2.brand_norm": "sony",
     "table1.category_norm": "headphones", "table2.category_norm": "audio headphones",
     "table1.modelno_norm": "wh1000xm4", "table2.modelno_norm": "wh1000xm4",
     "table1.price": 348.0, "table2.price": 329.99},
    {"table1.title_norm": "apple iphone 13 128gb blue",
     "table2.title_norm": "samsung galaxy s21 ultra 256gb phantom black",
     "table1.brand_norm": "apple", "table2.brand_norm": "samsung",
     "table1.category_norm": "cell phones", "table2.category_norm": "cell phones",
     "table1.modelno_norm": "mlpf3", "table2.modelno_norm": "smg998",
     "table1.price": 799.0, "table2.price": 1199.0},
])
demo["word_overlap"] = [len(set(a.split()) & set(b.split())) / len(set(a.split()) | set(b.split()))
                         for a, b in zip(demo["table1.title_norm"], demo["table2.title_norm"])]
demo["brand_match"] = (demo["table1.brand_norm"] == demo["table2.brand_norm"]).astype(int)
demo["category_match"] = (demo["table1.category_norm"] == demo["table2.category_norm"]).astype(int)
demo["price_diff"] = (demo["table1.price"] - demo["table2.price"]).abs()
demo["price_ratio"] = demo["table1.price"] / (demo["table2.price"] + 1e-8)
demo["has_both_prices"] = 1
demo["len_title_diff"] = (demo["table1.title_norm"].str.len() - demo["table2.title_norm"].str.len()).abs()

p_demo, yhat_demo = score_dataframe(demo, loaded, reload_embedder, DEVICE)
for i, (pp, yy) in enumerate(zip(p_demo, yhat_demo)):
    print(f"Par {i+1}: P(match)={pp:.3f} -> {'MATCH' if yy else 'NO-MATCH'}")

---
## 8. Conclusion

Implementamos y validamos una **solucion funcional de product matching** para Walmart-Amazon:
un clasificador supervisado sobre un espacio de features hibrido (estructural + texto clasico +
embeddings de transformer), con computo acelerado en GPU (CUDA). Comparamos tres modelos,
ajustamos el umbral bajo desbalance, evaluamos con metricas apropiadas (F1, PR-AUC, ROC-AUC) y
**justificamos cada bloque de senal mediante un ablation**. El modelo final queda **persistido y
se recarga reproduciendo las metricas**, listo para integrarse en el siguiente entregable.
